# Food Freshness and Quality Classification Using Deep Learning

**Architecture:** EfficientNetB0 transfer learning + fine-tuning  
**Dataset:** Freshness44 — 53,616 images, 22 produce categories  
**Classes:** Fresh / Rotten  
**Runtime:** Google Colab with T4 GPU

Run the cells from top to bottom. If the Colab runtime disconnects, reconnect, remount Drive, and rerun setup/data cells. Checkpoints are stored in Drive.

## 1. Select the GPU runtime

In Google Colab, select **Runtime → Change runtime type → T4 GPU** before running the notebook.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
import shutil
import zipfile
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, ConfusionMatrixDisplay
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))

## 2. Configure paths

Expected Drive location: `/content/drive/MyDrive/Freshness44/freshness44.zip`

The extracted dataset in Drive is expected at `/content/drive/MyDrive/Freshness44/data/Freshness44`. The ZIP is copied to Colab local storage for faster training.

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/Freshness44"
ZIP_PATH = f"{DRIVE_ROOT}/freshness44.zip"
DRIVE_DATASET_ROOT = f"{DRIVE_ROOT}/data/Freshness44"

LOCAL_ZIP = "/content/freshness44.zip"
LOCAL_DATASET_ROOT = "/content/Freshness44/Freshness44"

BEST_BASELINE = f"{DRIVE_ROOT}/best_model.keras"
BEST_FINETUNED = f"{DRIVE_ROOT}/best_finetuned_model.keras"
FINAL_MODEL = f"{DRIVE_ROOT}/Food_Freshness_FINAL.keras"

assert os.path.exists(ZIP_PATH), f"Dataset ZIP not found: {ZIP_PATH}"
print("Drive dataset ZIP found.")

In [ ]:
# Copy the ZIP to local Colab storage only if needed
if not os.path.exists(LOCAL_ZIP):
    shutil.copy2(ZIP_PATH, LOCAL_ZIP)
    print("ZIP copied to local storage.")
else:
    print("Local ZIP already exists.")

print(f"ZIP size: {os.path.getsize(LOCAL_ZIP)/(1024**3):.2f} GB")

In [ ]:
# Extract locally if not already extracted
if not os.path.isdir(LOCAL_DATASET_ROOT):
    os.makedirs(os.path.dirname(LOCAL_DATASET_ROOT), exist_ok=True)
    with zipfile.ZipFile(LOCAL_ZIP, "r") as archive:
        archive.extractall("/content")
    print("Dataset extracted.")
else:
    print("Local extracted dataset already exists.")

assert os.path.isdir(LOCAL_DATASET_ROOT), f"Dataset folder not found: {LOCAL_DATASET_ROOT}"
print("Dataset root:", LOCAL_DATASET_ROOT)

## 3. Build and verify the image dataframe

Folder names encode both the produce type and class, e.g. `Apple_Fresh` and `Apple_Rotten`. The original `Grape` and `Grapes` folders are kept separate as provided by the dataset.

In [ ]:
records = []
valid_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

for folder in sorted(os.listdir(LOCAL_DATASET_ROOT)):
    folder_path = os.path.join(LOCAL_DATASET_ROOT, folder)
    if not os.path.isdir(folder_path):
        continue

    if folder.endswith("_Fresh"):
        freshness, produce = "Fresh", folder[:-6]
    elif folder.endswith("_Rotten"):
        freshness, produce = "Rotten", folder[:-7]
    else:
        continue

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(valid_extensions):
            records.append({
                "image_path": os.path.join(folder_path, filename),
                "produce": produce,
                "freshness": freshness
            })

df = pd.DataFrame(records)
print("Total images:", len(df))
print("Produce categories:", df["produce"].nunique())
print("Classes:", sorted(df["freshness"].unique()))
display(df.head())

In [ ]:
assert len(df) == 53616, f"Expected 53,616 images, found {len(df)}"
assert df["produce"].nunique() == 22, f"Expected 22 produce categories, found {df['produce'].nunique()}"
assert set(df["freshness"].unique()) == {"Fresh", "Rotten"}
assert df["image_path"].map(os.path.isfile).all(), "At least one image path does not exist."
print("Dataset verification PASSED.")

In [ ]:
print("Freshness distribution:")
display(df["freshness"].value_counts().rename_axis("class").to_frame("images"))
print("Images by produce:")
display(df["produce"].value_counts().sort_index().rename_axis("produce").to_frame("images"))

## 4. Stratified train / validation / test split

The split is 80% train, 10% validation, and 10% test. Stratification uses the combination of produce category and freshness label.

In [ ]:
df["stratify_label"] = df["produce"] + "_" + df["freshness"]

train_df, temp_df = train_test_split(
    df, test_size=0.20, random_state=SEED,
    stratify=df["stratify_label"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=SEED,
    stratify=temp_df["stratify_label"]
)

print(f"Train: {len(train_df):,}")
print(f"Validation: {len(val_df):,}")
print(f"Test: {len(test_df):,}")
print(f"Total: {len(train_df) + len(val_df) + len(test_df):,}")

In [ ]:
for split_name, split_df in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    print(f"\n{split_name} class proportions:")
    print(split_df["freshness"].value_counts(normalize=True).round(4))

## 5. TensorFlow input pipeline

Images are resized to 224 × 224. `tf.ensure_shape` avoids image-rank inconsistencies during decoding/resizing.

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 64
AUTOTUNE = tf.data.AUTOTUNE
LABEL_MAP = {"Fresh": 0, "Rotten": 1}

def create_dataset(dataframe, shuffle=False):
    paths = dataframe["image_path"].to_numpy()
    labels = dataframe["freshness"].map(LABEL_MAP).to_numpy(dtype=np.float32)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    def load_image(path, label):
        image_bytes = tf.io.read_file(path)
        image = tf.image.decode_image(
            image_bytes, channels=3, expand_animations=False
        )
        image = tf.ensure_shape(image, [None, None, 3])
        image = tf.image.resize(image, IMG_SIZE)
        image = tf.cast(image, tf.float32)
        return image, label

    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(2000, seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = create_dataset(train_df, shuffle=True)
val_ds = create_dataset(val_df)
test_ds = create_dataset(test_df)
print("TensorFlow datasets are ready.")

## 6. Data augmentation

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
], name="data_augmentation")

## 7. Build the EfficientNetB0 model

The ImageNet-pretrained backbone is frozen for the initial transfer-learning stage.

In [ ]:
base_model = tf.keras.applications.EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)
base_model.trainable = False

model = tf.keras.Sequential([
    tf.keras.Input(shape=(224, 224, 3)),
    data_augmentation,
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1, activation="sigmoid")
], name="Food_Freshness_EfficientNetB0")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
)
model.summary()

## 8. Initial transfer-learning training

This stage trains the classification head. The best validation-accuracy checkpoint is saved to Drive.

In [ ]:
baseline_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        BEST_BASELINE, monitor="val_accuracy", mode="max",
        save_best_only=True, verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=2,
        restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.2, patience=1,
        min_lr=1e-6, verbose=1
    )
]

baseline_history = model.fit(
    train_ds, validation_data=val_ds,
    epochs=3, callbacks=baseline_callbacks
)

## 9. Fine-tune the upper EfficientNetB0 layers

Load the best baseline checkpoint, unfreeze the upper 40 backbone layers, and use a smaller learning rate. Training is allowed up to 50 epochs, but early stopping may stop sooner. The best validation checkpoint is retained.

In [ ]:
model = tf.keras.models.load_model(BEST_BASELINE)
base_model = model.layers[2]
base_model.trainable = True

for layer in base_model.layers[:-40]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
)

print("Trainable backbone layers:", sum(layer.trainable for layer in base_model.layers))

In [ ]:
finetune_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        BEST_FINETUNED, monitor="val_accuracy", mode="max",
        save_best_only=True, verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=4,
        restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.2, patience=2,
        min_lr=1e-7, verbose=1
    )
]

finetune_history = model.fit(
    train_ds, validation_data=val_ds,
    epochs=50, callbacks=finetune_callbacks
)

## 10. Load the best fine-tuned checkpoint

In [ ]:
assert os.path.exists(BEST_FINETUNED), "Fine-tuned checkpoint not found; run the fine-tuning cell first."
model = tf.keras.models.load_model(BEST_FINETUNED)
print("Loaded best checkpoint:", BEST_FINETUNED)

## 11. Final held-out test evaluation

Use the test set only for final reporting—not for selecting epochs, thresholds, or model architecture.

In [ ]:
test_results = model.evaluate(test_ds, verbose=1)
print(f"Test loss:     {test_results[0]:.4f}")
print(f"Test accuracy: {test_results[1] * 100:.2f}%")
print(f"Test AUC:      {test_results[2]:.4f}")

## 12. Classification report and ROC-AUC

In [ ]:
y_true = test_df["freshness"].map(LABEL_MAP).to_numpy()
y_prob = model.predict(test_ds, verbose=1).ravel()
y_pred = (y_prob >= 0.5).astype(int)

print(classification_report(
    y_true, y_pred, target_names=["Fresh", "Rotten"], digits=4
))
print(f"ROC-AUC: {roc_auc_score(y_true, y_prob):.4f}")

## 13. Confusion matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)
print(cm)

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Fresh", "Rotten"]
).plot()
plt.title("Food Freshness - Confusion Matrix")
plt.show()

## 14. Training curves

In [ ]:
def plot_history(history, title):
    h = history.history

    plt.figure(figsize=(8, 5))
    plt.plot(h["accuracy"], label="Training Accuracy")
    plt.plot(h["val_accuracy"], label="Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(f"{title} - Accuracy")
    plt.legend()
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(h["loss"], label="Training Loss")
    plt.plot(h["val_loss"], label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{title} - Loss")
    plt.legend()
    plt.grid(True)
    plt.show()

plot_history(baseline_history, "Transfer Learning")
plot_history(finetune_history, "Fine-Tuning")

## 15. Save the final model to Google Drive

This copies the best fine-tuned checkpoint. The source checkpoint remains unchanged.

In [ ]:
if not os.path.exists(BEST_FINETUNED):
    raise FileNotFoundError("best_finetuned_model.keras was not found.")

shutil.copy2(BEST_FINETUNED, FINAL_MODEL)
print("Final model saved to:", FINAL_MODEL)

## 16. Predict one image

Set `IMAGE_PATH` to a real image path, then call `predict_image(IMAGE_PATH)`. The confidence is the model probability assigned to the predicted class.

In [ ]:
def predict_image(image_path, model=model):
    image_bytes = tf.io.read_file(image_path)
    image = tf.image.decode_image(
        image_bytes, channels=3, expand_animations=False
    )
    image = tf.ensure_shape(image, [None, None, 3])
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)

    probability_rotten = float(
        model.predict(tf.expand_dims(image, 0), verbose=0)[0][0]
    )

    if probability_rotten >= 0.5:
        label, confidence = "Rotten", probability_rotten
    else:
        label, confidence = "Fresh", 1.0 - probability_rotten

    plt.figure(figsize=(5, 5))
    plt.imshow(tf.cast(image, tf.uint8))
    plt.axis("off")
    plt.title(f"{label} — Confidence: {confidence * 100:.2f}%")
    plt.show()
    return label, confidence

# Example:
# IMAGE_PATH = "/content/example.jpg"
# predict_image(IMAGE_PATH)

## 17. Previously recorded completed-run results

These are the recorded results from the completed experiment, not values calculated automatically by this summary cell.

| Metric | Recorded result |
|---|---:|
| Best fine-tuning epoch | 9 |
| Training accuracy | 98.09% |
| Validation accuracy | 98.43% |
| Held-out test accuracy | 98.56% |
| Held-out test AUC | 99.92% |
| Test loss | 0.0368 |

Best checkpoint: `best_finetuned_model.keras`

The notebook's training cells reproduce the workflow; training may produce different results if the software/runtime, data, split, or random behavior changes.